Tonny Talukder 043222000510050 :
MOVIE SEARCH ENGINE PROJECT

In [17]:
import pandas as pd
import numpy as np
import re
import nltk

from collections import defaultdict, Counter

from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

In [18]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [19]:
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [20]:
DATA_PATH = '/content/drive/MyDrive/CSE-Data-Mining-Warehouse-Lab-426/Search Engine Project/tmdb_5000_movies.csv'

movies = pd.read_csv(DATA_PATH)

In [21]:
movies = movies[['title', 'overview']]
movies.dropna(inplace=True)
movies.reset_index(drop=True, inplace=True)

### Displaying the first few rows of the dataset to confirm loading:

In [22]:
display(movies.head())

,title,overview
0,Avatar,"In the 22nd century, a paraplegic Marine is di..."
1,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha..."
2,Spectre,A cryptic message from Bond’s past sends him o...
3,The Dark Knight Rises,Following the death of District Attorney Harve...
4,John Carter,"John Carter is a war-weary, former military ca..."


In [23]:
stop_words = set(stopwords.words('english'))
ps = PorterStemmer()

In [24]:
def preprocess(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z0-9\s]', '', text)

    words = text.split()

    processed = []
    for w in words:
        if w not in stop_words:
            processed.append(ps.stem(w))

    return processed

In [25]:
movies['processed'] = movies['overview'].apply(preprocess)

In [26]:
inverted_index = defaultdict(dict)

for doc_id, words in enumerate(movies['processed']):
    freq = Counter(words)

    for word, count in freq.items():
        inverted_index[word][doc_id] = count

In [27]:
def and_search(query):
    words = preprocess(query)

    result_sets = []

    for w in words:
        if w in inverted_index:
            result_sets.append(set(inverted_index[w].keys()))

    if not result_sets:
        return []

    return list(set.intersection(*result_sets))

In [28]:
def or_search(query):
    words = preprocess(query)

    result = set()

    for w in words:
        if w in inverted_index:
            result.update(inverted_index[w].keys())

    return list(result)

In [29]:
def rank_documents(query, docs):
    words = preprocess(query)

    scores = {}

    for doc_id in docs:
        score = 0
        for w in words:
            score += inverted_index[w].get(doc_id, 0)

        scores[doc_id] = score

    return sorted(scores.items(), key=lambda x: x[1], reverse=True)

In [30]:
def show_results(ranked_docs, top_n=10):
    print("\n===== SEARCH RESULTS =====\n")

    for i, (doc_id, score) in enumerate(ranked_docs[:top_n]):
        print("----------------------------")
        print(f"Rank: {i+1}")
        print(f"Title: {movies.iloc[doc_id]['title']}")
        print(f"Score: {score}")
        print(f"Snippet: {movies.iloc[doc_id]['overview'][:200]}")

In [31]:
query = input("Enter your search query: ")
mode = input("AND / OR: ").upper()

if mode == "AND":
    docs = and_search(query)
else:
    docs = or_search(query)

ranked = rank_documents(query, docs)
show_results(ranked)

Enter your search query: Action Movie
AND / OR: or

===== SEARCH RESULTS =====

----------------------------
Rank: 1
Title: Jackass 3D
Score: 5
Snippet: Jackass 3D is a 3-D film and the third movie of the Jackass series. It follows the same premise as the first two movies, as well as the TV series. It is a compilation of various pranks, stunts and ski
----------------------------
Rank: 2
Title: Disaster Movie
Score: 5
Snippet: In DISASTER MOVIE, the filmmaking team behind the hits "Scary Movie," "Date Movie," "Epic Movie" and "Meet The Spartans" this time puts its unique, inimitable stamp on one of the biggest and most bloa
----------------------------
Rank: 3
Title: Rugrats Go Wild
Score: 3
Snippet: Rugrats Go Wild is a 2003 crossover animated film, with two animated Nickelodeon television series Rugrats and The Wild Thornberrys.The film was produced by Klasky Csupo and released in theaters on Ju
----------------------------
Rank: 4
Title: Grindhouse
Score: 3
Snippet: Two full length 